# 🚀 赛博少年（CyberLoRA）— 一键训练脚本

> 📌 **运行前必读**
> 1. 点击菜单栏 **「代码执行程序」→「更改运行时类型」** → 选择 **T4 GPU**
> 2. 准备好 **15-30 张个人照片** 放在本地文件夹中
> 3. 按顺序依次执行每个单元格（Shift+Enter）
> 4. 总耗时约 **20-45 分钟**

---
## 📦 Step 1：环境安装（约 2-3 分钟）

In [ ]:
# 挂载 Google Drive（用于持久化存储训练结果）
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/cyberlora'
os.makedirs(WORK_DIR, exist_ok=True)
%cd {WORK_DIR}
print('✅ 工作目录准备完成:', WORK_DIR)

In [ ]:
# 安装 Kohya_SS 及其依赖
!git clone --depth 1 https://github.com/bmaltais/kohya_ss.git
%cd kohya_ss

!apt-get update -qq && apt-get install -y -qq python3-tk > /dev/null 2>&1
!pip install -q --upgrade pip
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q xformers --index-url https://download.pytorch.org/whl/cu118
!pip install -q -r requirements.txt
!pip install -q accelerate==0.25.0
!pip install -q bitsandbytes
!pip install -q huggingface_hub

# 安装 wd14-tagger（自动打标）
!pip install -q onnxruntime-gpu onnx pillow

print('\n✅ 环境安装完成！')

---
## 🖼️ Step 2：上传训练图片（约 1 分钟）

> 🎯 执行下面单元格后，点击出现的「选择文件」按钮，
> **一次性选中你的 15-30 张照片**上传（支持 JPG/PNG）

In [ ]:
from google.colab import files
import shutil

TRAIN_DIR = f'{WORK_DIR}/train_data/100_cyberboy'
REG_DIR = f'{WORK_DIR}/reg_data/1_cyberboy'
os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(REG_DIR, exist_ok=True)

print('🔔 请选择 15-30 张个人照片上传...')
uploaded = files.upload()

# 移动到训练目录
for filename in uploaded.keys():
    shutil.move(filename, os.path.join(TRAIN_DIR, filename))

print(f'\n✅ 已上传 {len(uploaded)} 张照片到训练目录')

---
## 🔧 Step 3：图片预处理（统一尺寸）

In [ ]:
from PIL import Image

TARGET_SIZE = 1024

for fname in os.listdir(TRAIN_DIR):
    fpath = os.path.join(TRAIN_DIR, fname)
    if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue
    try:
        img = Image.open(fpath).convert('RGB')
        w, h = img.size
        # 等比缩放短边到 1024，居中裁剪
        scale = TARGET_SIZE / min(w, h)
        new_w, new_h = int(w * scale), int(h * scale)
        img = img.resize((new_w, new_h), Image.LANCZOS)
        left = (new_w - TARGET_SIZE) // 2
        top = (new_h - TARGET_SIZE) // 2
        img = img.crop((left, top, left + TARGET_SIZE, top + TARGET_SIZE))
        img.save(fpath, quality=95)
    except Exception as e:
        print(f'⚠️ 跳过 {fname}: {e}')

print(f'✅ 图片已统一处理为 {TARGET_SIZE}×{TARGET_SIZE}')

---
## 🏷️ Step 4：自动打标（wd14-tagger，约 1 分钟）

> 自动为每张图片生成文字描述标签

In [ ]:
!mkdir -p /content/models

# 下载 wd14-tagger 模型
!wget -q -O /content/models/wd14_tagger.onnx \
  https://huggingface.co/SmilingWolf/wd-v1-4-convnextv2-tagger-v2/resolve/main/model.onnx
!wget -q -O /content/models/wd14_tags.csv \
  https://huggingface.co/SmilingWolf/wd-v1-4-convnextv2-tagger-v2/resolve/main/selected_tags.csv

# 批量打标
import subprocess

tag_script = f"""
from pathlib import Path
import onnxruntime as ort
import numpy as np
from PIL import Image
import csv

TRAIN_DIR = Path('{TRAIN_DIR}')
MODEL_PATH = '/content/models/wd14_tagger.onnx'
TAGS_PATH = '/content/models/wd14_tags.csv'
THRESHOLD = 0.35

# 加载标签列表
with open(TAGS_PATH, 'r') as f:
    reader = csv.reader(f)
    tags = [row[1] for row in reader][1:]

session = ort.InferenceSession(MODEL_PATH, providers=['CUDAExecutionProvider'])

for img_path in sorted(TRAIN_DIR.glob('*.jpg')) + sorted(TRAIN_DIR.glob('*.png')):
    img = Image.open(img_path).convert('RGBA')
    # 转换为模型输入格式
    target_size = 448
    img = img.resize((target_size, target_size), Image.LANCZOS)
    img_np = np.array(img, dtype=np.float32)[:, :, :3]
    img_np = img_np[:, :, ::-1]  # RGB to BGR
    img_np = np.expand_dims(img_np, 0)
    
    output = session.run(None, {{'input_1': img_np}})[0]
    
    result_tags = []
    for i, prob in enumerate(output[0]):
        if prob > THRESHOLD:
            result_tags.append(tags[i])
    
    # 写 txt 文件（触发词 + 自动标签）
    txt_path = img_path.with_suffix('.txt')
    # 过滤掉面部特征描述，避免过拟合
    face_keywords = ['face', 'eyes', 'nose', 'mouth', 'eyebrows', 'lip', 'chin', 'forehead', 'jaw']
    filtered_tags = [t for t in result_tags if not any(fk in t.lower() for fk in face_keywords)]
    
    with open(txt_path, 'w') as f:
        f.write('cyberboy, ' + ', '.join(filtered_tags))

print(f'已处理 {{len(list(TRAIN_DIR.glob("*.jpg"))) + len(list(TRAIN_DIR.glob("*.png")))}} 张图片')
"""

with open('/content/tag_images.py', 'w') as f:
    f.write(tag_script)

!python /content/tag_images.py
print('\n✅ 自动打标完成！请检查生成的 .txt 文件')

---
## ⚙️ Step 5：配置训练参数

In [ ]:
import json

config = {
    "pretrained_model_name_or_path": "stabilityai/stable-diffusion-xl-base-1.0",
    "v2": False,
    "v_parameterization": False,
    "train_data_dir": TRAIN_DIR,
    "reg_data_dir": REG_DIR,
    "output_dir": f"{WORK_DIR}/output",
    "output_name": "cyberboy_sdxl",
    "save_model_as": "safetensors",
    "resolution": "1024,1024",
    "train_batch_size": 1,
    "max_train_epochs": 8,
    "network_module": "networks.lora",
    "network_dim": 32,
    "network_alpha": 16,
    "learning_rate": 1e-4,
    "unet_lr": 1e-4,
    "text_encoder_lr": 5e-5,
    "lr_scheduler": "cosine",
    "optimizer_type": "AdamW8bit",
    "mixed_precision": "bf16",
    "save_precision": "bf16",
    "save_every_n_epochs": 4,
    "caption_extension": ".txt",
    "max_token_length": 225,
    "enable_bucket": True,
    "min_bucket_reso": 512,
    "max_bucket_reso": 1536,
    "noise_offset": 0.1,
    "xformers": True,
    "sdpa": False,
    "cache_latents": True,
    "cache_latents_to_disk": True,
    "dataset_config": "",
    "shuffle_caption": False,
    "keep_tokens": 1,
    "sample_every_n_steps": 0
}

config_path = f'{WORK_DIR}/kohya_config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

os.makedirs(f"{WORK_DIR}/output", exist_ok=True)
print(f'✅ 配置文件已保存: {config_path}')

---
## 🏋️ Step 6：开始训练（约 15-40 分钟）

> 🔥 这是最关键的步骤！训练期间请保持浏览器窗口活跃，避免 Colab 断连。

In [ ]:
train_script = f"""
import subprocess, json, sys, os

CONFIG = '{config_path}'

with open(CONFIG, 'r') as f:
    cfg = json.load(f)

cmd = [
    sys.executable, 'sd-scripts/train_network.py',
    '--pretrained_model_name_or_path', cfg['pretrained_model_name_or_path'],
    '--train_data_dir', cfg['train_data_dir'],
    '--output_dir', cfg['output_dir'],
    '--output_name', cfg['output_name'],
    '--save_model_as', cfg['save_model_as'],
    '--resolution', cfg['resolution'],
    '--train_batch_size', str(cfg['train_batch_size']),
    '--max_train_epochs', str(cfg['max_train_epochs']),
    '--network_module', cfg['network_module'],
    '--network_dim', str(cfg['network_dim']),
    '--network_alpha', str(cfg['network_alpha']),
    '--learning_rate', str(cfg['learning_rate']),
    '--unet_lr', str(cfg['unet_lr']),
    '--text_encoder_lr', str(cfg['text_encoder_lr']),
    '--lr_scheduler', cfg['lr_scheduler'],
    '--optimizer_type', cfg['optimizer_type'],
    '--mixed_precision', cfg['mixed_precision'],
    '--save_precision', cfg['save_precision'],
    '--save_every_n_epochs', str(cfg['save_every_n_epochs']),
    '--caption_extension', cfg['caption_extension'],
    '--max_token_length', str(cfg['max_token_length']),
    '--noise_offset', str(cfg['noise_offset']),
    '--xformers',
    '--cache_latents',
    '--cache_latents_to_disk',
    '--enable_bucket',
    '--min_bucket_reso', str(cfg['min_bucket_reso']),
    '--max_bucket_reso', str(cfg['max_bucket_reso']),
]

print('🚀 训练开始...举杯☕，稍等片刻')
result = subprocess.run(cmd, cwd='{WORK_DIR}/kohya_ss')
print(f'\\n训练结束，返回码: {{result.returncode}}')
"""

with open(f'{WORK_DIR}/run_train.py', 'w') as f:
    f.write(train_script)

!cd {WORK_DIR}/kohya_ss && python ../run_train.py

---
## 📥 Step 7：下载训练好的 LoRA 模型

In [ ]:
# 复制到 Google Drive 持久化存储
DRIVE_DIR = '/content/drive/MyDrive/CyberLoRA'
os.makedirs(DRIVE_DIR, exist_ok=True)

!cp {WORK_DIR}/output/*.safetensors {DRIVE_DIR}/ 2>/dev/null
!cp {WORK_DIR}/output/*.json {DRIVE_DIR}/ 2>/dev/null

print(f'✅ LoRA 模型已保存到 Google Drive: {DRIVE_DIR}')
print(f'\n📁 文件列表:')
!ls -lh {DRIVE_DIR}/

# 也可以直接下载到本地
from google.colab import files as colab_files
for f in os.listdir(f'{WORK_DIR}/output/'):
    if f.endswith('.safetensors'):
        print(f'下载: {f}')
        colab_files.download(f'{WORK_DIR}/output/{f}')

---
## ✅ 完成！下一步

1. 📥 下载 `.safetensors` 文件到本地
2. 🔧 放入 Stable Diffusion WebUI 的 `models/Lora/` 目录
3. 🎨 生成时在 Prompt 中加入 `cyberboy` 触发词
4. 📸 参考下方 Prompt 模板测试效果

---
### 🧪 推荐测试 Prompt

| 场景 | Prompt |
|:---|:---|
| 棚拍肖像 | `cyberboy, 1boy, portrait, studio lighting, bokeh, detailed face` |
| 赛博朋克 | `cyberboy, 1boy, Tokyo street, neon lights, night, cyberpunk, rain` |
| 科幻宇航员 | `cyberboy, 1boy, astronaut, in spacesuit, Mars surface, cinematic` |
| 商务正装 | `cyberboy, 1boy, wearing black suit, office, professional, natural light` |
| 户外运动 | `cyberboy, 1boy, surfing, ocean waves, sunset, dynamic pose, splashing water` |
| 古风侠客 | `cyberboy, 1boy, ancient Chinese warrior, armor, temple background, ink painting style` |
| 咖啡日常 | `cyberboy, 1boy, sitting in cafe, reading book, warm lighting, cozy atmosphere` |
| 未来战士 | `cyberboy, 1boy, sci-fi soldier, mechanical armor, futuristic city, hdr, 8k` |

**推理参数建议**：Sampler: DPM++ 2M Karras, Steps: 25, CFG Scale: 7, LoRA Weight: 0.7-0.85